# DESeq2 Reanalysis Validation

Validate DESeq2 reanalysis results against published data using LFC-based gene matching.

**Problem**: Different genome annotation versions produce different gene IDs, preventing direct comparison.

**Solution**: Genes with identical expression produce identical log2 fold changes. We match genes by LFC correlation, which is annotation-agnostic.

## Preparing Publication Data

Publications report DEG data in various formats. Use an LLM to convert to standard CSV.

### Step 1: Identify Source Data
- Supplementary Excel files (most common)
- PDF tables in supplementary materials
- Figures with gene lists (requires OCR)

### Step 2: LLM Prompt Template
Copy this prompt to Claude or ChatGPT:

---
I have DEG data from a publication. Convert to CSV with exactly these columns:
- gene_id: gene identifier as written (preserve exact format)
- log2fc: log2 fold change value
- description: gene name/function (optional, can be empty)

Output ONLY the CSV, no explanations. Use comma delimiter.

[PASTE YOUR DATA HERE]

---

### Step 3: Common Format Issues
| Publication Format | How to Handle |
|-------------------|---------------|
| Excel with multiple sheets | Specify which sheet contains DEGs |
| Fold change (not log2) | Ask LLM to convert: log2fc = log2(fold_change) |
| Separate up/down tables | Combine; down-regulated should be negative |
| P-value but no LFC | Cannot use - need LFC values |
| Gene symbols only | Need systematic IDs for matching |

### Step 4: Save as CSV
Save LLM output as `publication_data.csv` in this directory.

## Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore', message='.*narwhals.*')

import pandas as pd
import numpy as np
from scipy import stats
import altair as alt
from pathlib import Path

# Enable interactive charts in notebook
alt.data_transformers.enable('default', max_rows=None)

## Configuration

In [ ]:
# === CONFIGURE THESE ===
# For Galaxy JupyterLite: use history dataset numbers
# For local Jupyter: use file paths

DESEQ2_HISTORY_ID = 638      # Galaxy history dataset number for DESeq2 TSV
PUBLICATION_HISTORY_ID = 639 # Galaxy history dataset number for publication CSV (upload wang24_invivo_publication.csv)

# Or use file paths for local Jupyter (set history IDs to None)
DESEQ2_FILE = None  # e.g., "../wang24_PRJNA1086003/analysis/deseq2_in_vivo.tsv"
PUBLICATION_FILE = None  # e.g., "../wang24_PRJNA1086003/wang24_invivo_publication.csv"

COMPARISON_NAME = "Wang et al. (2024) In Vivo vs Reanalysis"

# Optional filters
PADJ_THRESHOLD = None  # e.g., 0.05 to filter significant only
LFC_THRESHOLD = None   # e.g., 1.0 to filter |log2FC| > 1

## Load Data from Galaxy History

In [ ]:
# Load data from Galaxy history or local files
import io

try:
    import gxy
    # Galaxy JupyterLite - gxy.get() returns file path
    deseq2_path = await gxy.get(DESEQ2_HISTORY_ID)
    publication_path = await gxy.get(PUBLICATION_HISTORY_ID)
    print(f"Galaxy paths: {deseq2_path}, {publication_path}")
except ImportError:
    # Local Jupyter - use file paths from config
    deseq2_path = DESEQ2_FILE
    publication_path = PUBLICATION_FILE
    print(f"Local files: {deseq2_path}, {publication_path}")

## Functions

In [ ]:
def load_deseq2(path):
    """Load DESeq2 output TSV (handles with/without header)."""
    if path is None:
        raise ValueError("No DESeq2 data provided.")
    
    # Check if file has header by testing if second column name is numeric
    df = pd.read_csv(path, sep='\t', nrows=1)
    try:
        float(df.columns[1])
        # No header - first row is data
        df = pd.read_csv(path, sep='\t', header=None,
                        names=['Gene_ID', 'baseMean', 'log2FoldChange', 'lfcSE', 'stat', 'pvalue', 'padj'])
    except ValueError:
        # Has header
        df = pd.read_csv(path, sep='\t')
        if df.columns[0] != 'Gene_ID':
            df = df.rename(columns={df.columns[0]: 'Gene_ID'})
    return df

def load_publication(path):
    """Load prepared publication CSV."""
    if path is None:
        raise ValueError("No publication data provided.")
    return pd.read_csv(path)

In [4]:
def match_genes_by_lfc(deseq2_df, pub_df, tolerance=0.1, auto_direction=True):
    """
    Match genes between datasets using LFC correlation.

    When annotation versions differ, gene IDs don't match directly.
    But genes with identical expression produce identical LFCs.
    """
    results = []
    pub_lfcs = pub_df['log2fc'].values
    pub_genes = pub_df['gene_id'].values

    # Try both directions if auto_direction enabled
    directions = [1, -1] if auto_direction else [1]
    best_direction = 1
    best_matches = 0

    for direction in directions:
        matches = 0
        for i, (pub_gene, pub_lfc) in enumerate(zip(pub_genes, pub_lfcs)):
            target_lfc = pub_lfc * direction
            # Find closest match in DESeq2 data
            diffs = np.abs(deseq2_df['log2FoldChange'] - target_lfc)
            min_idx = diffs.idxmin()
            if diffs[min_idx] < tolerance:
                matches += 1
        if matches > best_matches:
            best_matches = matches
            best_direction = direction

    # Build mapping with best direction
    for pub_gene, pub_lfc in zip(pub_genes, pub_lfcs):
        target_lfc = pub_lfc * best_direction
        diffs = np.abs(deseq2_df['log2FoldChange'] - target_lfc)
        min_idx = diffs.idxmin()
        if diffs[min_idx] < tolerance:
            results.append({
                'source_gene': pub_gene,
                'source_lfc': pub_lfc,
                'target_gene': deseq2_df.loc[min_idx, 'Gene_ID'],
                'target_lfc': deseq2_df.loc[min_idx, 'log2FoldChange'] * best_direction,
                'lfc_diff': diffs[min_idx]
            })

    mapping_df = pd.DataFrame(results).sort_values('lfc_diff')
    return mapping_df, best_direction

In [5]:
def calculate_metrics(mapping_df):
    """Calculate validation metrics."""
    src = mapping_df['source_lfc']
    tgt = mapping_df['target_lfc']

    r2 = stats.pearsonr(src, tgt)[0] ** 2
    spearman = stats.spearmanr(src, tgt)[0]
    direction_agree = np.mean(np.sign(src) == np.sign(tgt)) * 100
    mean_diff = mapping_df['lfc_diff'].mean()

    return {
        'n_mapped': len(mapping_df),
        'pearson_r2': r2,
        'spearman_r': spearman,
        'direction_agreement': direction_agree,
        'mean_lfc_diff': mean_diff
    }

In [6]:
def plot_validation(mapping_df, name, metrics):
    """Generate interactive scatter + Bland-Altman plots with Altair."""

    # Prepare data with Bland-Altman columns
    plot_df = mapping_df.copy()
    plot_df['mean_lfc'] = (plot_df['source_lfc'] + plot_df['target_lfc']) / 2
    plot_df['diff_lfc'] = plot_df['source_lfc'] - plot_df['target_lfc']

    # Calculate regression for scatter plot
    slope, intercept, r, p, se = stats.linregress(plot_df['source_lfc'], plot_df['target_lfc'])
    x_range = [plot_df['source_lfc'].min(), plot_df['source_lfc'].max()]
    reg_df = pd.DataFrame({
        'x': x_range,
        'y_reg': [slope * x + intercept for x in x_range],
        'y_identity': x_range
    })

    # Bland-Altman stats
    mean_diff = plot_df['diff_lfc'].mean()
    std_diff = plot_df['diff_lfc'].std()
    loa_upper = mean_diff + 1.96 * std_diff
    loa_lower = mean_diff - 1.96 * std_diff

    # Scatter plot with regression
    scatter = alt.Chart(plot_df).mark_circle(size=60, opacity=0.7).encode(
        x=alt.X('source_lfc:Q', title='Publication log2FC'),
        y=alt.Y('target_lfc:Q', title='Reanalysis log2FC'),
        tooltip=['source_gene', 'target_gene',
                 alt.Tooltip('source_lfc:Q', format='.3f'),
                 alt.Tooltip('target_lfc:Q', format='.3f')]
    ).properties(
        title=f'{name} (R\u00b2 = {metrics["pearson_r2"]:.4f})',
        width=400, height=400
    )

    # Regression line
    reg_line = alt.Chart(reg_df).mark_line(color='red', strokeWidth=2).encode(
        x='x:Q', y='y_reg:Q'
    )

    # Identity line (y=x)
    identity_line = alt.Chart(reg_df).mark_line(
        color='black', strokeDash=[5, 5], opacity=0.5
    ).encode(x='x:Q', y='y_identity:Q')

    scatter_chart = scatter + reg_line + identity_line

    # Bland-Altman plot
    ba_scatter = alt.Chart(plot_df).mark_circle(size=60, opacity=0.7).encode(
        x=alt.X('mean_lfc:Q', title='Mean log2FC'),
        y=alt.Y('diff_lfc:Q', title='Difference (Pub - Reanalysis)'),
        tooltip=['source_gene', 'target_gene',
                 alt.Tooltip('mean_lfc:Q', format='.3f'),
                 alt.Tooltip('diff_lfc:Q', format='.3f')]
    ).properties(
        title='Bland-Altman Agreement',
        width=400, height=400
    )

    # Mean and LOA lines
    mean_line = alt.Chart(pd.DataFrame({'y': [mean_diff]})).mark_rule(
        color='red', strokeWidth=2
    ).encode(y='y:Q')

    upper_loa = alt.Chart(pd.DataFrame({'y': [loa_upper]})).mark_rule(
        color='gray', strokeDash=[5, 5]
    ).encode(y='y:Q')

    lower_loa = alt.Chart(pd.DataFrame({'y': [loa_lower]})).mark_rule(
        color='gray', strokeDash=[5, 5]
    ).encode(y='y:Q')

    ba_chart = ba_scatter + mean_line + upper_loa + lower_loa

    # Combine horizontally
    combined = alt.hconcat(scatter_chart, ba_chart).resolve_scale(
        color='independent'
    )

    return combined

## Run Validation

In [ ]:
# Load and parse data
deseq2_df = load_deseq2(deseq2_path)
pub_df = load_publication(publication_path)

# Apply filters if set
if PADJ_THRESHOLD:
    deseq2_df = deseq2_df[deseq2_df['padj'] < PADJ_THRESHOLD]
if LFC_THRESHOLD:
    deseq2_df = deseq2_df[np.abs(deseq2_df['log2FoldChange']) > LFC_THRESHOLD]

print(f"DESeq2 genes: {len(deseq2_df)}")
print(f"Publication genes: {len(pub_df)}")

# Match genes
mapping_df, direction = match_genes_by_lfc(deseq2_df, pub_df)
print(f"\nMatched genes: {len(mapping_df)}")
print(f"Direction: {'same' if direction == 1 else 'reversed (auto-corrected)'}")

# Calculate metrics
metrics = calculate_metrics(mapping_df)
print(f"\n=== Validation Metrics ===")
for k, v in metrics.items():
    print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")

## Interactive Validation Plot

In [ ]:
# Generate and display interactive chart
chart = plot_validation(mapping_df, COMPARISON_NAME, metrics)
chart